# Notebook 07 — Clone × Process Optimization Simulation

## Goal

Notebook07 moves from clone-only selection to clone-process pair selection.

Earlier notebooks asked:

> Which clone should we select?

Notebook07 asks:

> Which clone performs best under which process condition?

---

## Core idea

A clone's performance is not fixed.

Its final productivity, stability, and quality may depend on:

- media richness
- nutrient limitation
- feeding strategy
- metabolic stress
- aggregation risk

This notebook simulates conservative process-condition effects and evaluates whether process-aware clone-process pairing can improve selection.

---

## Strategy

We define several virtual process conditions:

- baseline
- rich_media
- nutrient_limited
- balanced_feed

For each clone × process pair, we estimate:

- process-adjusted late qP
- process-adjusted qP drop
- process-adjusted late aggregation
- process-adjusted utility

Then we select the best clone-process pairs.

In [1]:
# --------------------------------------------------
# Load prediction table from Notebook03
# --------------------------------------------------
import pandas as pd
import numpy as np
from pathlib import Path

scenario = "legacy"   # "legacy" or "optimized"
n_clones = 5000

PRED_PATH = Path("../data/synthetic/processed") / (
    f"predictions_03b_qp_{n_clones}_{scenario}.csv"
)

print("Loading:", PRED_PATH)

df = pd.read_csv(PRED_PATH)

print("Shape:", df.shape)
display(df.head())

required_cols = [
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_stable_prob",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
]

missing = [c for c in required_cols if c not in df.columns]
print("Missing required columns:", missing)

assert len(missing) == 0, f"Missing required columns: {missing}"

Loading: ../data/synthetic/processed/predictions_03b_qp_5000_legacy.csv
Shape: (1000, 20)


,clone_id,pred_qp_drop,pred_late_qp,pred_late_agg,pred_stable_prob,pred_stable_label,true_qp_drop,true_late_qp,true_late_agg,true_stable_label,pred_super_prob,pred_aggr_prob,true_is_aggressive,rescue_upside_qp,rescue_stability_band,rescue_quality_band,rescue_aggressive_penalty,rescue_not_already_pass,pred_rescue_score,pred_rescue_label
0,CLONE_1502,0.392653,2.728385e-08,9.545294,0.369596,0,0.111023,4.176362e-08,12.566994,1,0.000000,0.014588,0,0.000671,0.857824,0.809946,0.985223,0.508615,0.592822,1
1,CLONE_2587,0.496398,6.469268e-08,3.244036,0.193538,0,0.691891,3.691256e-08,0.918511,0,0.000000,0.000000,0,0.007506,0.512007,0.000000,1.000000,0.744142,0.274402,0
2,CLONE_2654,0.360346,3.089098e-08,7.252523,0.476211,0,0.009193,4.002863e-08,6.558486,1,0.004683,0.002842,0,0.001330,0.965513,0.643578,0.997121,0.365989,0.576372,0
3,CLONE_1056,0.363795,6.799885e-08,6.921204,0.478655,0,0.708930,3.026369e-08,7.990409,0,0.003250,0.000000,0,0.008111,0.954018,0.548916,1.000000,0.362719,0.549059,0
4,CLONE_0706,0.611245,1.741702e-07,8.346533,0.041611,0,0.740570,1.029717e-07,7.441547,0,0.019114,0.025802,0,0.027511,0.129184,0.956152,0.973863,0.947386,0.425159,0


Missing required columns: []


## Step 1 — Utility helpers

We define utility scores for evaluating clone quality.

Higher utility means:

- higher late qP
- lower qP drop
- lower aggregation

The true utility is used only for offline evaluation.

In [2]:
# --------------------------------------------------
# Step 1 — Utility helper functions
# --------------------------------------------------

def z(s):
    s = pd.Series(s).astype(float)
    return (s - s.mean()) / (s.std(ddof=0) + 1e-9)

def z01(s):
    s = pd.Series(s).astype(float)
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

def make_true_utility(df):
    return (
        1.0 * z(df["true_late_qp"])
        - 1.0 * z(df["true_qp_drop"])
        - 0.2 * z(df["true_late_agg"])
    )

def make_pred_utility(df, qp_col, drop_col, agg_col):
    return (
        1.0 * z(df[qp_col])
        - 1.0 * z(df[drop_col])
        - 0.2 * z(df[agg_col])
    )

def topk_overlap_by_id(true_ids, selected_ids, k=10):
    return len(set(true_ids) & set(selected_ids)) / k

df["true_util"] = make_true_utility(df)
df["baseline_score"] = make_pred_utility(
    df,
    qp_col="pred_late_qp",
    drop_col="pred_qp_drop",
    agg_col="pred_late_agg",
)

display(df[[
    "clone_id",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "pred_rescue_score",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
    "baseline_score",
]].head())

,clone_id,pred_late_qp,pred_qp_drop,pred_late_agg,pred_rescue_score,true_late_qp,true_qp_drop,true_late_agg,true_util,baseline_score
0,CLONE_1502,2.728385e-08,0.392653,9.545294,0.592822,4.176362e-08,0.111023,12.566994,1.031479,-0.194735
1,CLONE_2587,6.469268e-08,0.496398,3.244036,0.274402,3.691256e-08,0.691891,0.918511,-1.010771,-0.528996
2,CLONE_2654,3.089098e-08,0.360346,7.252523,0.576372,4.002863e-08,0.009193,6.558486,1.875600,0.313863
3,CLONE_1056,6.799885e-08,0.363795,6.921204,0.549059,3.026369e-08,0.708930,7.990409,-1.521727,0.394926
4,CLONE_0706,1.741702e-07,0.611245,8.346533,0.425159,1.029717e-07,0.740570,7.441547,-1.610143,-1.793905


## Step 2 — Define virtual process conditions

We define four conservative virtual process conditions.

These are not real bioreactor recipes.  
They are simplified process scenarios used to test clone × process interactions.

### Process conditions

- baseline: no change
- rich_media: modest qP gain, but possible stability / aggregation burden
- nutrient_limited: lower productivity, but improved stability and quality
- balanced_feed: small improvement across productivity, stability, and quality

The effect sizes are intentionally conservative.

In [3]:
# --------------------------------------------------
# Step 2 — Define conservative process conditions
# --------------------------------------------------
# Values represent baseline process effect before clone-specific sensitivity.
#
# qp_effect:
#   relative change in predicted late qP
#
# drop_effect:
#   absolute change in predicted qP drop
#   negative = improved stability
#   positive = worse stability
#
# agg_effect:
#   relative change in predicted aggregation
#   negative = reduced aggregation
#   positive = increased aggregation

process_conditions = {
    "baseline": {
        "qp_effect": 0.00,
        "drop_effect": 0.00,
        "agg_effect": 0.00,
    },
    "rich_media": {
        "qp_effect": 0.08,
        "drop_effect": 0.03,
        "agg_effect": 0.06,
    },
    "nutrient_limited": {
        "qp_effect": -0.04,
        "drop_effect": -0.08,
        "agg_effect": -0.08,
    },
    "balanced_feed": {
        "qp_effect": 0.04,
        "drop_effect": -0.04,
        "agg_effect": -0.04,
    },
}

pd.DataFrame(process_conditions).T

,qp_effect,drop_effect,agg_effect
baseline,0.00,0.00,0.00
rich_media,0.08,0.03,0.06
nutrient_limited,-0.04,-0.08,-0.08
balanced_feed,0.04,-0.04,-0.04


## Step 3 — Clone-specific process sensitivity

Different clones should respond differently to process changes.

We define conservative sensitivity proxies using existing predictions:

- rescue sensitivity:
  - higher rescue score means more process optimization potential

- instability sensitivity:
  - moderate qP drop may indicate room for stabilization
  - extreme instability should not be over-rewarded

- aggregation sensitivity:
  - moderate aggregation may be reducible by process tuning
  - extreme aggregation remains risky

This is still rule-based, not learned from real process data.

In [4]:
# --------------------------------------------------
# Step 3 — Clone-specific process sensitivity
# --------------------------------------------------

def triangular_band_score(x, low, high, peak=None):
    x = pd.Series(x).astype(float)
    if peak is None:
        peak = (low + high) / 2

    score = pd.Series(np.zeros(len(x)), index=x.index)

    left = (x >= low) & (x <= peak)
    right = (x > peak) & (x <= high)

    score.loc[left] = (x.loc[left] - low) / (peak - low + 1e-12)
    score.loc[right] = (high - x.loc[right]) / (high - peak + 1e-12)

    return score.clip(0, 1)

work = df.copy()

work["sens_rescue"] = z01(work["pred_rescue_score"])

work["sens_stability_band"] = triangular_band_score(
    work["pred_qp_drop"],
    low=0.20,
    peak=0.35,
    high=0.60,
)

work["sens_agg_band"] = triangular_band_score(
    work["pred_late_agg"],
    low=5.0,
    peak=9.0,
    high=16.0,
)

work["process_sensitivity"] = (
    0.55 * work["sens_rescue"]
    + 0.25 * work["sens_stability_band"]
    + 0.20 * work["sens_agg_band"]
)

print("=== Process sensitivity summary ===")
display(work["process_sensitivity"].describe())

display(work[[
    "clone_id",
    "pred_rescue_score",
    "pred_qp_drop",
    "pred_late_agg",
    "sens_rescue",
    "sens_stability_band",
    "sens_agg_band",
    "process_sensitivity",
]].sort_values("process_sensitivity", ascending=False).head(15))

=== Process sensitivity summary ===


count    1000.000000
mean        0.460771
std         0.161830
min         0.000000
25%         0.365763
50%         0.450289
75%         0.573155
max         0.986811
Name: process_sensitivity, dtype: float64

,clone_id,pred_rescue_score,pred_qp_drop,pred_late_agg,sens_rescue,sens_stability_band,sens_agg_band,process_sensitivity
180,CLONE_4625,1.000000,0.349658,8.747600,1.000000,0.997723,0.936900,0.986811
425,CLONE_4976,0.665526,0.354303,8.893272,0.665526,0.982786,0.973318,0.806400
438,CLONE_0178,0.677257,0.352025,8.685189,0.677257,0.991901,0.921297,0.804726
246,CLONE_2179,0.667900,0.359162,8.869076,0.667900,0.963351,0.967269,0.801637
898,CLONE_3393,0.690476,0.373651,8.857666,0.690476,0.905395,0.964416,0.798994
29,CLONE_0487,0.641968,0.360991,9.079421,0.641968,0.956038,0.988654,0.789822
511,CLONE_2520,0.681207,0.346905,8.398650,0.681207,0.979363,0.849663,0.789437
117,CLONE_2095,0.656849,0.366629,8.851583,0.656849,0.933484,0.962896,0.787217
463,CLONE_3317,0.649654,0.369479,9.063349,0.649654,0.922084,0.990950,0.786021
500,CLONE_3717,0.644151,0.348688,9.621402,0.644151,0.991251,0.911228,0.784341


## Step 4 — Build clone × process table

We expand the dataset so that each clone appears once per process condition.

This allows us to evaluate:

> clone A under baseline  
> clone A under rich media  
> clone A under nutrient limitation  
> clone A under balanced feed

The selection problem becomes:

> Which clone-process pair is best?

In [5]:
# --------------------------------------------------
# Step 4 — Expand clones into clone × process pairs
# --------------------------------------------------

rows = []

for process_name, effect in process_conditions.items():
    tmp = work.copy()
    tmp["process_condition"] = process_name
    tmp["base_qp_effect"] = effect["qp_effect"]
    tmp["base_drop_effect"] = effect["drop_effect"]
    tmp["base_agg_effect"] = effect["agg_effect"]
    rows.append(tmp)

pair_df = pd.concat(rows, ignore_index=True)

print("Clone-process table shape:", pair_df.shape)
print("Number of unique clones:", pair_df["clone_id"].nunique())
print("Process conditions:", pair_df["process_condition"].unique())

display(pair_df[[
    "clone_id",
    "process_condition",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "process_sensitivity",
]].head(12))

Clone-process table shape: (4000, 30)
Number of unique clones: 1000
Process conditions: <StringArray>
['baseline', 'rich_media', 'nutrient_limited', 'balanced_feed']
Length: 4, dtype: str


,clone_id,process_condition,pred_late_qp,pred_qp_drop,pred_late_agg,process_sensitivity
0,CLONE_1502,baseline,2.728385e-08,0.392653,9.545294,0.717819
1,CLONE_2587,baseline,6.469268e-08,0.496398,3.244036,0.254523
2,CLONE_2654,baseline,3.089098e-08,0.360346,7.252523,0.669285
3,CLONE_1056,baseline,6.799885e-08,0.363795,6.921204,0.634248
4,CLONE_0706,baseline,1.741702e-07,0.611245,8.346533,0.401164
5,CLONE_0107,baseline,9.355133e-08,0.369435,5.540337,0.502701
6,CLONE_0590,baseline,3.992501e-07,0.464922,6.153344,0.426218
7,CLONE_2469,baseline,5.995046e-08,0.355418,4.385842,0.468327
8,CLONE_2414,baseline,5.486997e-08,0.398138,4.385395,0.401131
9,CLONE_1601,baseline,1.288802e-07,0.369964,3.887100,0.450069


## Step 5 — Apply clone-specific process effects

Each process condition has a base effect.

The actual effect is modulated by clone-specific sensitivity.

This creates clone × process interaction:

- rich media may help high-rescue clones more
- nutrient limitation may help unstable or aggregation-prone clones
- balanced feed may provide moderate all-around improvement

The effects remain conservative to avoid unrealistic ranking flips.

In [6]:
# --------------------------------------------------
# Step 5 — Apply clone-specific process effects
# --------------------------------------------------

# Clone-specific modulation
# Kept conservative to prevent artificial over-promotion.
pair_df["response_multiplier"] = 0.50 + 0.50 * pair_df["process_sensitivity"]

# Rich media can increase productivity more in process-sensitive clones,
# but can also increase aggregation burden.
is_rich = pair_df["process_condition"] == "rich_media"

# Nutrient limitation helps stability / aggregation more for sensitive clones,
# but may reduce qP slightly.
is_limited = pair_df["process_condition"] == "nutrient_limited"

# Balanced feed gives smaller but safer improvements.
is_balanced = pair_df["process_condition"] == "balanced_feed"

# Default adjusted effects
pair_df["adj_qp_effect"] = pair_df["base_qp_effect"] * pair_df["response_multiplier"]
pair_df["adj_drop_effect"] = pair_df["base_drop_effect"] * pair_df["response_multiplier"]
pair_df["adj_agg_effect"] = pair_df["base_agg_effect"] * pair_df["response_multiplier"]

# Extra conservative risk adjustment:
# rich media aggregation penalty is stronger for high predicted aggregation clones
pair_df.loc[is_rich, "adj_agg_effect"] = (
    pair_df.loc[is_rich, "adj_agg_effect"]
    * (1.0 + 0.30 * z01(pair_df.loc[is_rich, "pred_late_agg"]))
)

# Nutrient limitation stability benefit is stronger for moderate instability
pair_df.loc[is_limited, "adj_drop_effect"] = (
    pair_df.loc[is_limited, "adj_drop_effect"]
    * (1.0 + 0.30 * pair_df.loc[is_limited, "sens_stability_band"])
)

# Balanced feed quality benefit is stronger for moderate aggregation
pair_df.loc[is_balanced, "adj_agg_effect"] = (
    pair_df.loc[is_balanced, "adj_agg_effect"]
    * (1.0 + 0.20 * pair_df.loc[is_balanced, "sens_agg_band"])
)

# Apply effects
pair_df["process_late_qp"] = (
    pair_df["pred_late_qp"] * (1.0 + pair_df["adj_qp_effect"])
)

pair_df["process_qp_drop"] = (
    pair_df["pred_qp_drop"] + pair_df["adj_drop_effect"]
).clip(lower=0.0)

pair_df["process_late_agg"] = (
    pair_df["pred_late_agg"] * (1.0 + pair_df["adj_agg_effect"])
).clip(lower=0.0)

display(pair_df[[
    "clone_id",
    "process_condition",
    "process_sensitivity",
    "adj_qp_effect",
    "adj_drop_effect",
    "adj_agg_effect",
    "pred_late_qp",
    "process_late_qp",
    "pred_qp_drop",
    "process_qp_drop",
    "pred_late_agg",
    "process_late_agg",
]].sort_values("process_sensitivity", ascending=False).head(20))

,clone_id,process_condition,process_sensitivity,adj_qp_effect,adj_drop_effect,adj_agg_effect,pred_late_qp,process_late_qp,pred_qp_drop,process_qp_drop,pred_late_agg,process_late_agg
2180,CLONE_4625,nutrient_limited,0.986811,-0.039736,-0.103260,-0.079472,5.496331e-06,5.277928e-06,0.349658,0.246399,8.747600,8.052407
3180,CLONE_4625,balanced_feed,0.986811,0.039736,-0.039736,-0.047182,5.496331e-06,5.714734e-06,0.349658,0.309922,8.747600,8.334871
1180,CLONE_4625,rich_media,0.986811,0.079472,0.029802,0.071217,5.496331e-06,5.933138e-06,0.349658,0.379461,8.747600,9.370576
180,CLONE_4625,baseline,0.986811,0.000000,0.000000,0.000000,5.496331e-06,5.496331e-06,0.349658,0.349658,8.747600,8.747600
3425,CLONE_4976,balanced_feed,0.806400,0.036128,-0.036128,-0.043161,7.952432e-08,8.239737e-08,0.354303,0.318175,8.893272,8.509432
425,CLONE_4976,baseline,0.806400,0.000000,0.000000,0.000000,7.952432e-08,7.952432e-08,0.354303,0.354303,8.893272,8.893272
1425,CLONE_4976,rich_media,0.806400,0.072256,0.027096,0.064983,7.952432e-08,8.527042e-08,0.354303,0.381399,8.893272,9.471180
2425,CLONE_4976,nutrient_limited,0.806400,-0.036128,-0.093560,-0.072256,7.952432e-08,7.665126e-08,0.354303,0.260744,8.893272,8.250680
1438,CLONE_0178,rich_media,0.804726,0.072189,0.027071,0.064591,1.352013e-07,1.449614e-07,0.352025,0.379096,8.685189,9.246171
438,CLONE_0178,baseline,0.804726,0.000000,0.000000,0.000000,1.352013e-07,1.352013e-07,0.352025,0.352025,8.685189,8.685189


## Step 6 — Compute clone-process utility

Now each clone-process pair receives a utility score.

The score uses process-adjusted predicted values:

- process_late_qp
- process_qp_drop
- process_late_agg

This score represents expected performance under a specific process condition.

In [7]:
# --------------------------------------------------
# Step 6 — Compute clone-process utility
# --------------------------------------------------

pair_df["process_pair_score"] = make_pred_utility(
    pair_df,
    qp_col="process_late_qp",
    drop_col="process_qp_drop",
    agg_col="process_late_agg",
)

display(pair_df[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_util",
]].sort_values("process_pair_score", ascending=False).head(15))

,clone_id,process_condition,process_pair_score,process_late_qp,process_qp_drop,process_late_agg,pred_rescue_score,pred_rescue_label,true_util
3180,CLONE_4625,balanced_feed,13.152088,0.000006,0.309922,8.334871,1.000000,1,3.402515
1180,CLONE_4625,rich_media,12.945818,0.000006,0.379461,9.370576,1.000000,1,3.402515
2180,CLONE_4625,nutrient_limited,12.751699,0.000005,0.246399,8.052407,1.000000,1,3.402515
180,CLONE_4625,baseline,12.281934,0.000005,0.349658,8.747600,1.000000,1,3.402515
3621,CLONE_3254,balanced_feed,10.555863,0.000004,0.269352,7.729004,0.795627,1,5.085175
2621,CLONE_3254,nutrient_limited,10.345043,0.000004,0.219280,7.488087,0.795627,1,5.085175
1621,CLONE_3254,rich_media,10.280105,0.000005,0.331122,8.557861,0.795627,1,5.085175
3986,CLONE_2776,balanced_feed,10.110515,0.000006,0.625426,6.934344,0.581181,0,28.727846
1986,CLONE_2776,rich_media,9.968873,0.000006,0.675395,7.513042,0.581181,0,28.727846
621,CLONE_3254,baseline,9.886782,0.000004,0.304649,8.056853,0.795627,1,5.085175


## Step 7 — Select best process condition per clone

For each clone, we select the process condition with the highest process-pair score.

This converts the problem from:

> clone × process pair ranking

back to:

> one best expected process condition per clone

In [8]:
# --------------------------------------------------
# Step 7 — Best process per clone
# --------------------------------------------------

best_pair_per_clone = (
    pair_df.sort_values("process_pair_score", ascending=False)
    .groupby("clone_id", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("Best-pair table shape:", best_pair_per_clone.shape)

print("\n=== Best process condition distribution ===")
display(best_pair_per_clone["process_condition"].value_counts())

display(best_pair_per_clone[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "baseline_score",
    "process_sensitivity",
    "pred_rescue_score",
    "pred_rescue_label",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]].sort_values("process_pair_score", ascending=False).head(15))

Best-pair table shape: (1000, 38)

=== Best process condition distribution ===


process_condition
nutrient_limited    993
balanced_feed         7
Name: count, dtype: int64

,clone_id,process_condition,process_pair_score,baseline_score,process_sensitivity,pred_rescue_score,pred_rescue_label,process_late_qp,process_qp_drop,process_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_4625,balanced_feed,13.152088,12.715822,0.986811,1.000000,1,5.714734e-06,0.309922,8.334871,1.017815e-05,0.440572,11.935039,3.402515
1,CLONE_3254,balanced_feed,10.555863,10.301253,0.764853,0.795627,1,4.373118e-06,0.269352,7.729004,8.830813e-06,0.000000,9.991042,5.085175
2,CLONE_2776,balanced_feed,10.110515,9.737404,0.427695,0.581181,0,5.526772e-06,0.625426,6.934344,7.851137e-05,0.742645,3.219493,28.727846
3,CLONE_0048,balanced_feed,9.869466,9.502905,0.463303,0.590936,1,5.387710e-06,0.578705,10.788158,1.974793e-05,0.563132,11.626145,6.514772
4,CLONE_4878,balanced_feed,9.745600,9.390940,0.741674,0.775698,1,4.655462e-06,0.413913,9.851228,4.980082e-06,0.663904,5.051639,0.768562
5,CLONE_2171,balanced_feed,8.712099,8.272231,0.527978,0.620918,1,5.149750e-06,0.667216,9.129939,5.684738e-06,0.522193,8.604411,1.493767
6,CLONE_3863,balanced_feed,6.851935,6.426997,0.515934,0.622235,1,4.381173e-06,0.692981,8.172536,2.970790e-06,0.715311,3.496560,-0.151446
7,CLONE_1619,nutrient_limited,6.736567,6.728141,0.275109,0.327163,0,2.204066e-06,0.200272,2.761672,2.551172e-06,0.107071,0.670928,2.733002
8,CLONE_0535,nutrient_limited,2.993879,2.607712,0.663373,0.631904,1,1.557847e-06,0.407944,8.393434,1.120280e-06,0.643965,7.358242,-0.758120
9,CLONE_0643,nutrient_limited,2.834334,2.576393,0.355840,0.424363,0,1.665035e-06,0.479255,5.863619,1.814575e-06,0.512626,6.048606,0.208288


## Step 8 — Compare baseline clone selection vs clone-process selection

We compare:

1. baseline Top10 selected by original predicted utility
2. clone-process Top10 selected by best process-pair score

This tells us whether process pairing changes clone selection.

In [9]:
# --------------------------------------------------
# Step 8 — Top10 comparison
# --------------------------------------------------

TOP_K = 10

baseline_top10 = (
    work.sort_values("baseline_score", ascending=False)
    .head(TOP_K)
    .copy()
)

process_pair_top10 = (
    best_pair_per_clone.sort_values("process_pair_score", ascending=False)
    .head(TOP_K)
    .copy()
)

baseline_ids = set(baseline_top10["clone_id"])
process_ids = set(process_pair_top10["clone_id"])

print("=== Top10 comparison ===")
print("Baseline rescue count:", int(baseline_top10["pred_rescue_label"].sum()))
print("Process-pair rescue count:", int(process_pair_top10["pred_rescue_label"].sum()))
print(f"Top10 overlap: {len(baseline_ids & process_ids)}/{TOP_K}")

print("\n=== Baseline Top10 ===")
display(baseline_top10[[
    "clone_id",
    "baseline_score",
    "pred_rescue_score",
    "pred_rescue_label",
    "pred_late_qp",
    "pred_qp_drop",
    "pred_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

print("\n=== Process-pair Top10 ===")
display(process_pair_top10[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "baseline_score",
    "process_sensitivity",
    "pred_rescue_score",
    "pred_rescue_label",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

=== Top10 comparison ===
Baseline rescue count: 7
Process-pair rescue count: 7
Top10 overlap: 10/10

=== Baseline Top10 ===


,clone_id,baseline_score,pred_rescue_score,pred_rescue_label,pred_late_qp,pred_qp_drop,pred_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
180,CLONE_4625,12.715822,1.000000,1,0.000005,0.349658,8.747600,0.000010,0.440572,11.935039,3.402515
621,CLONE_3254,10.301253,0.795627,1,0.000004,0.304649,8.056853,0.000009,0.000000,9.991042,5.085175
986,CLONE_2776,9.737404,0.581181,0,0.000005,0.653980,7.160908,0.000079,0.742645,3.219493,28.727846
505,CLONE_0048,9.502905,0.590936,1,0.000005,0.607971,11.159931,0.000020,0.563132,11.626145,6.514772
730,CLONE_4878,9.390940,0.775698,1,0.000004,0.448747,10.267458,0.000005,0.663904,5.051639,0.768562
524,CLONE_2171,8.272231,0.620918,1,0.000005,0.697776,9.473428,0.000006,0.522193,8.604411,1.493767
314,CLONE_1619,6.728141,0.327163,0,0.000002,0.257101,2.910100,0.000003,0.107071,0.670928,2.733002
63,CLONE_3863,6.426997,0.622235,1,0.000004,0.723299,8.474087,0.000003,0.715311,3.496560,-0.151446
666,CLONE_0535,2.607712,0.631904,1,0.000002,0.483759,8.991695,0.000001,0.643965,7.358242,-0.758120
23,CLONE_0643,2.576393,0.424363,0,0.000002,0.537552,6.199860,0.000002,0.512626,6.048606,0.208288



=== Process-pair Top10 ===


,clone_id,process_condition,process_pair_score,baseline_score,process_sensitivity,pred_rescue_score,pred_rescue_label,process_late_qp,process_qp_drop,process_late_agg,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_4625,balanced_feed,13.152088,12.715822,0.986811,1.000000,1,0.000006,0.309922,8.334871,0.000010,0.440572,11.935039,3.402515
1,CLONE_3254,balanced_feed,10.555863,10.301253,0.764853,0.795627,1,0.000004,0.269352,7.729004,0.000009,0.000000,9.991042,5.085175
2,CLONE_2776,balanced_feed,10.110515,9.737404,0.427695,0.581181,0,0.000006,0.625426,6.934344,0.000079,0.742645,3.219493,28.727846
3,CLONE_0048,balanced_feed,9.869466,9.502905,0.463303,0.590936,1,0.000005,0.578705,10.788158,0.000020,0.563132,11.626145,6.514772
4,CLONE_4878,balanced_feed,9.745600,9.390940,0.741674,0.775698,1,0.000005,0.413913,9.851228,0.000005,0.663904,5.051639,0.768562
5,CLONE_2171,balanced_feed,8.712099,8.272231,0.527978,0.620918,1,0.000005,0.667216,9.129939,0.000006,0.522193,8.604411,1.493767
6,CLONE_3863,balanced_feed,6.851935,6.426997,0.515934,0.622235,1,0.000004,0.692981,8.172536,0.000003,0.715311,3.496560,-0.151446
7,CLONE_1619,nutrient_limited,6.736567,6.728141,0.275109,0.327163,0,0.000002,0.200272,2.761672,0.000003,0.107071,0.670928,2.733002
8,CLONE_0535,nutrient_limited,2.993879,2.607712,0.663373,0.631904,1,0.000002,0.407944,8.393434,0.000001,0.643965,7.358242,-0.758120
9,CLONE_0643,nutrient_limited,2.834334,2.576393,0.355840,0.424363,0,0.000002,0.479255,5.863619,0.000002,0.512626,6.048606,0.208288


## Step 9 — Utility overlap and true performance

We evaluate whether clone-process selection better recovers truly good clones.

Utility overlap compares:

- true Top10 by true_util
- baseline selected Top10
- process-pair selected Top10

We also compare true performance averages.

In [10]:
# --------------------------------------------------
# Step 9 — Utility overlap and true performance
# --------------------------------------------------

true_top10 = work.sort_values("true_util", ascending=False).head(TOP_K)
true_top_ids = set(true_top10["clone_id"])

baseline_overlap = len(true_top_ids & set(baseline_top10["clone_id"])) / TOP_K
process_pair_overlap = len(true_top_ids & set(process_pair_top10["clone_id"])) / TOP_K

print("=== Utility overlap ===")
print(f"Baseline utility overlap:     {baseline_overlap:.3f}")
print(f"Process-pair utility overlap: {process_pair_overlap:.3f}")

summary = pd.DataFrame({
    "baseline_top10": baseline_top10[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
    "process_pair_top10": process_pair_top10[[
        "true_late_qp",
        "true_qp_drop",
        "true_late_agg",
        "true_util",
    ]].mean(),
})

summary["direction"] = [
    "higher is better",
    "lower is better",
    "lower is better",
    "higher is better",
]

display(summary)

=== Utility overlap ===
Baseline utility overlap:     0.500
Process-pair utility overlap: 0.500


,baseline_top10,process_pair_top10,direction
true_late_qp,0.000014,0.000014,higher is better
true_qp_drop,0.491142,0.491142,lower is better
true_late_agg,6.800211,6.800211,lower is better
true_util,4.802436,4.802436,higher is better


## Step 10 — Process condition diagnostics

This step checks which process conditions are selected among top candidates.

This helps interpret whether the process model behaves reasonably.

Expected behavior:

- rich_media may appear for productivity-driven clones
- nutrient_limited may appear for unstable or aggregation-prone clones
- balanced_feed should often be a safe compromise

In [11]:
# --------------------------------------------------
# Step 10 — Process condition diagnostics
# --------------------------------------------------

print("=== Process condition distribution in Top10 ===")
display(process_pair_top10["process_condition"].value_counts())

print("\n=== Mean predicted process-adjusted outcomes by selected process ===")
display(process_pair_top10.groupby("process_condition")[[
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "process_pair_score",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]].mean())

print("\n=== Rescue label distribution by selected process ===")
display(pd.crosstab(
    process_pair_top10["process_condition"],
    process_pair_top10["pred_rescue_label"],
    rownames=["process_condition"],
    colnames=["pred_rescue_label"],
))

=== Process condition distribution in Top10 ===


process_condition
balanced_feed       7
nutrient_limited    3
Name: count, dtype: int64


=== Mean predicted process-adjusted outcomes by selected process ===


,process_late_qp,process_qp_drop,process_late_agg,process_pair_score,true_late_qp,true_qp_drop,true_late_agg,true_util
process_condition,,,,,,,,
balanced_feed,0.000005,0.508217,8.705726,9.856795,0.000019,0.521108,7.703476,6.548742
nutrient_limited,0.000002,0.362490,5.672908,4.188260,0.000002,0.421221,4.692592,0.727723



=== Rescue label distribution by selected process ===


pred_rescue_label,0,1
process_condition,,
balanced_feed,1,6
nutrient_limited,2,1


## Step 11 — Production Top3 guardrail

Top10 can include exploratory clone-process pairs.

However, Top3 should remain conservative because these represent production-like candidates.

Here we select Top3 from process-pair Top10 but require:

- acceptable predicted qP drop
- acceptable predicted aggregation

In [12]:
# --------------------------------------------------
# Step 11 — Conservative Top3 production candidates
# --------------------------------------------------

TOP3_MAX_DROP = 0.45
TOP3_MAX_AGG = 14.0

top3_pool = process_pair_top10[
    (process_pair_top10["process_qp_drop"] <= TOP3_MAX_DROP) &
    (process_pair_top10["process_late_agg"] <= TOP3_MAX_AGG)
].copy()

process_pair_top3 = (
    top3_pool.sort_values("process_pair_score", ascending=False)
    .head(3)
    .copy()
)

print("Top3 candidates:", len(process_pair_top3))

display(process_pair_top3[[
    "clone_id",
    "process_condition",
    "process_pair_score",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "pred_rescue_score",
    "pred_rescue_label",
    "true_late_qp",
    "true_qp_drop",
    "true_late_agg",
    "true_util",
]])

Top3 candidates: 3


,clone_id,process_condition,process_pair_score,process_late_qp,process_qp_drop,process_late_agg,pred_rescue_score,pred_rescue_label,true_late_qp,true_qp_drop,true_late_agg,true_util
0,CLONE_4625,balanced_feed,13.152088,0.000006,0.309922,8.334871,1.000000,1,0.000010,0.440572,11.935039,3.402515
1,CLONE_3254,balanced_feed,10.555863,0.000004,0.269352,7.729004,0.795627,1,0.000009,0.000000,9.991042,5.085175
4,CLONE_4878,balanced_feed,9.745600,0.000005,0.413913,9.851228,0.775698,1,0.000005,0.663904,5.051639,0.768562


## Step 12 — Action recommendation table

The previous steps selected clone × process pairs.

This step converts those results into a practical decision-support output.

Instead of only reporting scores, we generate an action table with:

- clone ID
- selected process condition
- recommended action
- expected performance after process pairing
- expected improvement versus baseline
- rescue / pass status
- confidence tier
- biological rationale

This is the table a CLD or CDMO scientist would actually review before deciding which clones should move forward, which clones should receive process optimization, and which clones should remain exploratory.

In [13]:
# --------------------------------------------------
# Step 12 — Action recommendation table
# --------------------------------------------------

def find_existing_df(candidate_names):
    for name in candidate_names:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            print(f"Using dataframe: {name}")
            return globals()[name].copy(), name
    raise NameError(f"None of these dataframes were found: {candidate_names}")


process_top10_df, _ = find_existing_df([
    "process_pair_top10",
    "process_top10",
    "top10_process",
    "top10_pairs",
    "top10",
])

process_top3_df, _ = find_existing_df([
    "process_pair_top3",
    "production_top3",
    "process_top3",
    "top3_process",
    "top3_pairs",
    "top3",
])

action = process_top10_df.copy()

top3_ids = set(process_top3_df["clone_id"])

# Expected improvement vs baseline
action["delta_late_qp"] = action["process_late_qp"] - action["pred_late_qp"]
action["delta_late_qp_pct"] = action["delta_late_qp"] / (action["pred_late_qp"].abs() + 1e-12)

action["delta_qp_drop"] = action["process_qp_drop"] - action["pred_qp_drop"]
action["delta_late_agg"] = action["process_late_agg"] - action["pred_late_agg"]

# Decision role
action["decision_role"] = np.where(
    action["clone_id"].isin(top3_ids),
    "Top3 production candidate",
    "Top10 process-development candidate",
)

def recommended_action(row):
    rescue = int(row.get("pred_rescue_label", 0))
    process = row["process_condition"]
    drop = row["process_qp_drop"]
    agg = row["process_late_agg"]

    if row["decision_role"] == "Top3 production candidate":
        return "Advance as production candidate"

    if rescue == 1 and process == "balanced_feed":
        return "Process-rescue candidate: test balanced-feed optimization"

    if rescue == 1 and process == "nutrient_limited":
        return "Process-rescue candidate: test nutrient-limited risk reduction"

    if rescue == 1 and process == "rich_media":
        return "Exploratory rescue: rich-media boost with risk monitoring"

    if drop > 0.45:
        return "Monitor stability risk before advancement"

    if agg > 14.0:
        return "Monitor aggregation risk before advancement"

    return "Process-development candidate"

def confidence_tier(row):
    rescue = int(row.get("pred_rescue_label", 0))
    stable_prob = row.get("pred_stable_prob", np.nan)
    drop = row["process_qp_drop"]
    agg = row["process_late_agg"]

    if rescue == 0 and stable_prob >= 0.65 and drop <= 0.35 and agg <= 10:
        return "High"

    if rescue == 0 and stable_prob >= 0.50 and drop <= 0.45 and agg <= 14:
        return "Medium"

    if rescue == 1 and drop <= 0.45 and agg <= 14:
        return "Medium-rescue"

    return "Low / exploratory"

def rationale(row):
    parts = []

    if int(row.get("pred_rescue_label", 0)) == 1:
        parts.append("rescue-labeled clone with process optimization potential")
    else:
        parts.append("non-rescue clone selected by process-aware ranking")

    process = row["process_condition"]

    if process == "balanced_feed":
        parts.append("balanced-feed selected to preserve productivity while reducing risk")
    elif process == "nutrient_limited":
        parts.append("nutrient-limited condition selected for stability/quality risk reduction")
    elif process == "rich_media":
        parts.append("rich-media condition selected for productivity boost")
    elif process == "baseline":
        parts.append("baseline remains optimal")

    if row["delta_late_qp_pct"] > 0.02:
        parts.append("expected qP improvement")

    if row["delta_qp_drop"] < -0.02:
        parts.append("expected qP-drop reduction")

    if row["delta_late_agg"] < -0.2:
        parts.append("expected aggregation reduction")

    return "; ".join(parts)

action["recommended_action"] = action.apply(recommended_action, axis=1)
action["confidence_tier"] = action.apply(confidence_tier, axis=1)
action["rationale"] = action.apply(rationale, axis=1)

action_recommendation_table = action[[
    "clone_id",
    "decision_role",
    "process_condition",
    "recommended_action",
    "confidence_tier",
    "process_pair_score",
    "pred_rescue_label",
    "pred_rescue_score",
    "pred_stable_prob",
    "process_late_qp",
    "process_qp_drop",
    "process_late_agg",
    "delta_late_qp_pct",
    "delta_qp_drop",
    "delta_late_agg",
    "rationale",
]].sort_values(
    ["decision_role", "process_pair_score"],
    ascending=[True, False]
).reset_index(drop=True)

print("=== Action recommendation table ===")
print("Shape:", action_recommendation_table.shape)

display(action_recommendation_table)

Using dataframe: process_pair_top10
Using dataframe: process_pair_top3
=== Action recommendation table ===
Shape: (10, 16)


,clone_id,decision_role,process_condition,recommended_action,confidence_tier,process_pair_score,pred_rescue_label,pred_rescue_score,pred_stable_prob,process_late_qp,process_qp_drop,process_late_agg,delta_late_qp_pct,delta_qp_drop,delta_late_agg,rationale
0,CLONE_2776,Top10 process-development candidate,balanced_feed,Monitor stability risk before advancement,Low / exploratory,10.110515,0,0.581181,0.089303,0.000006,0.625426,6.934344,0.028554,-0.028554,-0.226564,non-rescue clone selected by process-aware ran...
1,CLONE_0048,Top10 process-development candidate,balanced_feed,Process-rescue candidate: test balanced-feed o...,Low / exploratory,9.869466,1,0.590936,0.125575,0.000005,0.578705,10.788158,0.029266,-0.029266,-0.371773,rescue-labeled clone with process optimization...
2,CLONE_2171,Top10 process-development candidate,balanced_feed,Process-rescue candidate: test balanced-feed o...,Low / exploratory,8.712099,1,0.620918,0.099117,0.000005,0.667216,9.129939,0.030560,-0.030560,-0.343489,rescue-labeled clone with process optimization...
3,CLONE_3863,Top10 process-development candidate,balanced_feed,Process-rescue candidate: test balanced-feed o...,Low / exploratory,6.851935,1,0.622235,0.067827,0.000004,0.692981,8.172536,0.030319,-0.030319,-0.301552,rescue-labeled clone with process optimization...
4,CLONE_1619,Top10 process-development candidate,nutrient_limited,Process-development candidate,High,6.736567,0,0.327163,0.684381,0.000002,0.200272,2.761672,-0.025502,-0.056829,-0.148428,non-rescue clone selected by process-aware ran...
5,CLONE_0535,Top10 process-development candidate,nutrient_limited,Process-rescue candidate: test nutrient-limite...,Medium-rescue,2.993879,1,0.631904,0.123425,0.000002,0.407944,8.393434,-0.033267,-0.075816,-0.598262,rescue-labeled clone with process optimization...
6,CLONE_0643,Top10 process-development candidate,nutrient_limited,Monitor stability risk before advancement,Low / exploratory,2.834334,0,0.424363,0.107337,0.000002,0.479255,5.863619,-0.027117,-0.058298,-0.336241,non-rescue clone selected by process-aware ran...
7,CLONE_4625,Top3 production candidate,balanced_feed,Advance as production candidate,Medium-rescue,13.152088,1,1.000000,0.451737,0.000006,0.309922,8.334871,0.039736,-0.039736,-0.412729,rescue-labeled clone with process optimization...
8,CLONE_3254,Top3 production candidate,balanced_feed,Advance as production candidate,Medium-rescue,10.555863,1,0.795627,0.571229,0.000004,0.269352,7.729004,0.035297,-0.035297,-0.327849,rescue-labeled clone with process optimization...
9,CLONE_4878,Top3 production candidate,balanced_feed,Advance as production candidate,Medium-rescue,9.745600,1,0.775698,0.200522,0.000005,0.413913,9.851228,0.034833,-0.034833,-0.416230,rescue-labeled clone with process optimization...


### Interpretation — action recommendation table

This table is the practical output of Notebook07.

The key transition is:

> clone ranking → clone × process action recommendation

Each row now tells us:

- which clone to consider
- which process condition is recommended
- whether the clone is a direct production candidate or process-development candidate
- whether it is a rescue clone
- what improvement is expected versus baseline
- why the recommendation was made

This is still a simulation output.  
The process effects are not yet learned from real media/feed/process data.

However, this table is the first version of a CDMO-facing decision output.

In [14]:
# --------------------------------------------------
# Save action recommendation table
# --------------------------------------------------

if "action_recommendation_table" not in globals():
    raise NameError(
        "action_recommendation_table is not defined. "
        "Run Step 12 — Action recommendation table first."
    )

OUT_ACTION = Path("../reports") / f"notebook07_action_recommendations_{n_clones}_{scenario}.csv"
OUT_ACTION.parent.mkdir(parents=True, exist_ok=True)

action_recommendation_table.to_csv(OUT_ACTION, index=False)

print("Saved action recommendation table:", OUT_ACTION)
print("Shape:", action_recommendation_table.shape)

Saved action recommendation table: ../reports/notebook07_action_recommendations_5000_legacy.csv
Shape: (10, 16)


## Final interpretation of Notebook07

Notebook07 extends the CLD pipeline from clone-only decision-making to clone × process decision-making.

The main output is no longer only a ranked clone list.  
It is an action recommendation table:

> clone ID → recommended process condition → predicted post-process profile → decision role → rationale

This is an important SDL transition.

Notebook07 now produces three levels of interpretation:

1. **Clone ranking**
   - Which clones are predicted to perform well?

2. **Clone × process pairing**
   - Which process condition is expected to improve each clone?

3. **Action recommendation**
   - Which clone should be advanced, rescued, monitored, or treated as exploratory?

---

## What this notebook demonstrates

1. Clone ranking can change under different virtual process conditions.
2. Process response is clone-specific rather than uniform.
3. Rescue clones may become more competitive under selected process conditions.
4. Top10 can be used as an exploratory clone-process candidate pool.
5. Top3 remains protected by conservative production guardrails.
6. The final action table translates model outputs into scientist-readable recommendations.

---

## Practical output

The action recommendation table is the key output of this notebook.

It summarizes:

- `clone_id`
- `decision_role`
- `process_condition`
- `recommended_action`
- `confidence_tier`
- predicted post-process qP, qP drop, and aggregation
- expected change versus baseline
- biological / process rationale

This makes Notebook07 closer to a decision-support layer rather than only an analysis notebook.

---

## Scientific basis

The simulation assumptions are intentionally conservative.

They are motivated by established CHO cell culture observations:

- Fed-batch feeding strategy and nutrient control influence cell growth, productivity, and product quality.
- Dynamic feeding strategies using real-time glucose monitoring have been applied in CHO culture for monoclonal antibody production.
- Media/feed combinations can affect titer and glycosylated antibody quality.
- Continuous or controlled feeding can reduce metabolic byproducts and improve antibody expression.
- Product quality attributes such as glycosylation are sensitive to culture conditions.

---

## References

1. Zalai et al. investigated using specific productivity as a physiological control parameter in mammalian cell culture processes.
2. Domján et al. developed Raman-based dynamic feeding strategies using real-time glucose monitoring in adalimumab-producing CHO cell culture.
3. Lu et al. demonstrated automated dynamic fed-batch process and media optimization for mammalian cell culture.
4. Xiao et al. showed that continuous feeding can reduce metabolic byproduct generation and improve antibody expression in CHO-K1 cultures.
5. Gyorgypal et al. evaluated the impact of media and feed combinations on glycosylated monoclonal antibody production using CHO cells.

---

## Current limitation

This notebook is still a simulation.

The process effects are not learned from real LC/GC, spent media, Raman, media composition, feed schedule, or bioreactor data.

Therefore, results should be interpreted as:

> A hypothesis-generating clone × process simulation framework

rather than a validated bioprocess optimizer.

---

## Next step

The next major step is Notebook08:

> Reference-based calibration of synthetic CLD / CHO process realism.

Notebook08 should compare the synthetic data generator against public CHO reference ranges such as NISTCHO-related data and published CHO process literature.

The goal is to determine whether the simulator produces realistic ranges for:

- titer
- VCD
- viability
- aggregation
- productivity decay
- early-to-late predictability

This will help move the pipeline from a well-structured synthetic prototype toward a more scientifically defensible CLD decision-support system.